In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from routingpy import OSRM
import time

# Hospital Location Cleaning and Demand Calculation

## Methodology

A total of 632 Pluvicto treatment centers were collected from the Novartis patient support site ([treatment center locator](https://us.pluvicto.com/treatment-center-locator)), which provides the names and locations (including address, city, state, and ZIP code) of each facility. Five treatment sites located in Hawaii, Alaska, and Puerto Rico were excluded from the dataset because they are outside the continental United States. Given that the distance between each treatment site and the production site is calculated in terms of vehicle driving time, these overseas territories would not be compatible with the current modeling framework. After exclusion, **627 treatment centers** were included in this project.

### Demand Estimation

The first step in preparing the data for the optimization problem is to estimate the demand for Pluvicto at each site. In practice, patient demand is typically known to the pharmaceutical company; however, due to the lack of proprietary data for this project, demand was estimated using three factors: **(1)** the type of treatment center, **(2)** the total bed size at each treatment center, and **(3)** the county-level incidence rate of late-stage prostate cancer where each treatment site is located.

The composite weight for each treatment center is calculated as follows:

$$
\text{Treatment Center Weight} = (\text{Center Type Weight} \times \text{Total Bed Count} \times \text{County-Level Late-Stage Prostate Cancer Incidence Rate})
$$

where:
- **Center Type Weight** is set to **5** for NCI-Designated Cancer Centers, **2** for academic hospitals, and **1** for community hospitals, consistent with prior research on the distribution of cancer patients across hospital types ([ASCO abstract](https://www.asco.org/abstracts-presentations/186938)).
- **Total Bed Count** for each site was obtained from the Medicare Hospital Provider Cost Report ([Medicare Hospital Provider Cost Report](https://catalog.data.gov/dataset/hospital-provider-cost-report)). For sites without a Medicare cost report, typically outpatient treatment centers exempt from reporting, bed size was imputed using the median bed count of all other treatment sites with reported Medicare bed counts.
- **County-Level Late-Stage Prostate Cancer Incidence Rate** was sourced from the NIH State Cancer Profile ([Zenodo dataset](https://zenodo.org/records/11098815)). For counties without reported incidence rates, the state average (if available) or national average was applied.

### Methodological Justification

This composite weighting scheme is methodologically justified, as it integrates three key determinants of a hospital's potential patient volume:

1. **Center Type Weight (Technology and Catchment Effect):** The 5:2:1 weight ratio captures both the advanced therapeutic capabilities and the strong regional or even national patient attraction effect associated with NCI-designated and academic centers.

2. **Total Bed Count (Capacity and Scale Effect):** This metric serves as a proxy for the hospital's physical infrastructure and overall service capacity.

3. **County-Level Incidence Rate (Local Demand Effect):** This variable reflects the size of the local patient pool available to the hospital, as patients tend to seek care within their county of residence.

The product of these three factors yields a relative measure of each hospital's potential Pluvicto demand. This composite weight is then used to proportionally allocate the estimated national Pluvicto demand across all hospitals.

### National Demand Estimation

The total demand for Pluvicto is estimated to be **250,000 vials per year**, which corresponds to the production capacity of Novartis's Indianapolis production site in 2024 ([Fierce Pharma report](https://www.fiercepharma.com/manufacturing/novartis-expands-pluvicto-manufacturing-footprint-fda-blessing-indianapolis#:~:text=With%20the%20addition%20of%20the%20Indianapolis%20site%2C%20Novartis,doses%20of%20radiotherapies%20annually%20in%202024%20and%20beyond.)). At that time, this was the only U.S. facility capable of producing Pluvicto. The underlying logic is consistent with established healthcare utilization frameworks that link provider capacity, technological sophistication, and local disease burden to observed patient distribution patterns.

In [28]:
pluvicto_hos = pd.read_excel("pluvicto_hospitals.xlsx")
state_cancer_profile = pd.read_csv("state_cancer_profiles_incidence.csv")
medicare_cost_report = pd.read_csv("CostReport_2023_Final.csv")

Here's a quick overview of the pluvicto treament site, state cancer profile, and Medicare Hospital Provider Cost Report

In [3]:
pluvicto_hos.head()

,Hospital (Novartis),Address,City,State,Zip,rpt_rec_num,Designation,LATITUDE,LONGITUDE
0,MONUMENT HEALTH RAPID CITY HOSPITAL - FAIRMONT...,353 FAIRMONT BLVD MEDICAL IMAGING/ NUCLEAR MED...,RAPID CITY,SD,57701,793743.0,1,44.079142,-103.215297
1,CHEYENNE REGIONAL MEDICAL CENTER,BLDG WEST CAMPUS FL 1 214 E 23RD ST,CHEYENNE,WY,82001-3748,791460.0,1,41.143175,-104.785212
2,BILLINGS CLINIC,2800 10TH AVE N,BILLINGS,MT,59101,777548.0,1,45.772835,-108.500117
3,ROCKY MOUNTAIN CANCER CENTERS LLC - BOULDER,4715 ARAPHOE AVE FL 1,BOULDER,CO,80303-1385,795451.0,1,40.001324,-105.233878
4,KAISER PERMANENTE ROCK CREEK,280 EXEMPLA CIR,LAFAYETTE,CO,80026,795452.0,1,39.996426,-105.101503


In [4]:
state_cancer_profile.head()

,full_locale,fips,age_adjusted_incidence_raterate_note___cases_per_100_000,lower_95pct_confidence_interval,upper_95pct_confidence_interval,ci_rankrank_note,lower_ci_ci_rank,upper_ci_ci_rank,average_annual_count,recent_trend,...,areatype,age,state_fips,measurement,locale_type,_extracted_at,url,percent_of_cases_with_late_stage,locale,state
0,US (SEER+NPCR)(1),0,442.3,442.0,442.6,NaN,NaN,NaN,1698328.0,stable,...,By County,All Ages,0,incd,national,2024-05-01T11:14:03.752827,https://statecancerprofiles.cancer.gov/inciden...,NaN,US,NaN
1,"Union County, Florida(6)",12125,1237.4,1165.6,1312.8,NaN,1.0,1.0,237.0,stable,...,By County,All Ages,12,incd,county,2024-05-01T11:14:03.752827,https://statecancerprofiles.cancer.gov/inciden...,NaN,Union County,Florida
2,"Palo Alto County, Iowa(7)",19147,658.1,591.1,731.1,NaN,1.0,6.0,82.0,rising,...,By County,All Ages,19,incd,county,2024-05-01T11:14:03.752827,https://statecancerprofiles.cancer.gov/inciden...,NaN,Palo Alto County,Iowa
3,"Treasure County, Montana(6)",30103,652.2,401.0,1007.4,NaN,1.0,55.0,7.0,stable,...,By County,All Ages,30,incd,county,2024-05-01T11:14:03.752827,https://statecancerprofiles.cancer.gov/inciden...,NaN,Treasure County,Montana
4,"Polk County, Texas(7)",48373,633.6,604.6,663.7,NaN,1.0,4.0,425.0,rising,...,By County,All Ages,48,incd,county,2024-05-01T11:14:03.752827,https://statecancerprofiles.cancer.gov/inciden...,NaN,Polk County,Texas


In [5]:
medicare_cost_report.head()

,rpt_rec_num,Provider CCN,Hospital Name,Street Address,City,State Code,Zip Code,County,Medicare CBSA Number,Rural Versus Urban,...,Net Income from Service to Patients,Total Other Income,Total Income,Total Other Expenses,Net Income,Cost To Charge Ratio,Net Revenue from Medicaid,Medicaid Charges,Net Revenue from Stand-Alone CHIP,Stand-Alone CHIP Charges
0,747534,110130,IRWIN COUNTY HOSPITAL,710 NORTH IRWIN AVENUE,OCILLA,GA,31774,IRWIN,99911.0,R,...,-1475853.0,227033.0,-1248820.0,NaN,-1248820.0,0.426683,30844.0,140289.0,13.0,212.0
1,748262,144042,LAKE BEHAVIORAL HOSPITAL,2615 WASHINGTON ST,WAUKEGAN,IL,60085,LAKE,29404.0,U,...,1198476.0,1434403.0,2632879.0,NaN,2632879.0,NaN,NaN,NaN,NaN,NaN
2,748457,43036,EVEREST REHABILITATION HOSPITAL BENT,4313 S PLEASANT CROSSING BLVD,ROGERS,AR,72758-1347,BENTON,22220.0,U,...,-997485.0,478980.0,-518505.0,NaN,-518505.0,NaN,NaN,NaN,NaN,NaN
3,748589,454155,OCEANS BEHAVIORAL HOSPITAL CORPUS CH,600 ELIZABETH ST BUILDING B 4TH FLO,CORPUS CHRISTI,TX,78404,NUECES,18580.0,U,...,-2183467.0,170.0,-2183297.0,NaN,-2183297.0,NaN,NaN,NaN,NaN,NaN
4,748617,144043,MONTROSE BEHAVIORAL HEALTH HOSPITAL,4720 NORTH CLARENDON AVENUE,CHICAGO,IL,60640-5122,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Given that the state name in Pluvicto treatment site is abbreviated, the first step is to create a cross walk between state abbreviated name and full name, so the information can be joined with other data set

In [29]:
# Dictionary mapping state abbreviations to full names
us_states = {
    'AL': 'Alabama',
    'AK': 'Alaska',
    'AZ': 'Arizona',
    'AR': 'Arkansas',
    'CA': 'California',
    'CO': 'Colorado',
    'CT': 'Connecticut',
    'DE': 'Delaware',
    'FL': 'Florida',
    'GA': 'Georgia',
    'HI': 'Hawaii',
    'ID': 'Idaho',
    'IL': 'Illinois',
    'IN': 'Indiana',
    'IA': 'Iowa',
    'KS': 'Kansas',
    'KY': 'Kentucky',
    'LA': 'Louisiana',
    'ME': 'Maine',
    'MD': 'Maryland',
    'MA': 'Massachusetts',
    'MI': 'Michigan',
    'MN': 'Minnesota',
    'MS': 'Mississippi',
    'MO': 'Missouri',
    'MT': 'Montana',
    'NE': 'Nebraska',
    'NV': 'Nevada',
    'NH': 'New Hampshire',
    'NJ': 'New Jersey',
    'NM': 'New Mexico',
    'NY': 'New York',
    'NC': 'North Carolina',
    'ND': 'North Dakota',
    'OH': 'Ohio',
    'OK': 'Oklahoma',
    'OR': 'Oregon',
    'PA': 'Pennsylvania',
    'RI': 'Rhode Island',
    'SC': 'South Carolina',
    'SD': 'South Dakota',
    'TN': 'Tennessee',
    'TX': 'Texas',
    'UT': 'Utah',
    'VT': 'Vermont',
    'VA': 'Virginia',
    'WA': 'Washington',
    'WV': 'West Virginia',
    'WI': 'Wisconsin',
    'WY': 'Wyoming'
}

# Convert all state abbreviation to full name
pluvicto_hos['State_Full'] = pluvicto_hos['State'].map(us_states)

Merge Medicare Cost Report to Pluvicto treatment site to get bed count information

In [30]:
# Merge the bed count information from Medicare Cost Report to Pluvicto treatment site on 'rpt_rec_num'
pluvicto_hos = pluvicto_hos.merge(
    medicare_cost_report[['rpt_rec_num', 'Number of Beds']], 
    on='rpt_rec_num', 
    how='left'
)

# Calculate median and round to nearest integer
median_beds = round(pluvicto_hos['Number of Beds'].median())

# Fill null values with the rounded median
pluvicto_hos['Number of Beds'] = pluvicto_hos['Number of Beds'].fillna(median_beds)


Merge Medicare Cost Report to Pluvicto treatment site to get county location information

In [31]:
# Merge the county name information from Medicare Cost Report to Pluvicto treatment site on 'rpt_rec_num'
pluvicto_hos = pluvicto_hos.merge(
    medicare_cost_report[['rpt_rec_num', 'County']], 
    on='rpt_rec_num', 
    how='left'
)

# Convert County column to first letter of each word capitalized
pluvicto_hos['County'] = pluvicto_hos['County'].str.title()

Filter out State Cancer Profile to get only late stage prostate cancer profile by county for all races and ages

In [32]:
# Filter the DataFrame with multiple conditions
state_cancer_profile_filtered = state_cancer_profile[
    (state_cancer_profile["stage"] == "Late Stage (Regional & Distant)") &
    (state_cancer_profile["race"] == "All Races (includes Hispanic)") &
    (state_cancer_profile["cancer"] == "Prostate") &
    (state_cancer_profile["age"] == "All Ages")
]

Merge State Cancer Profile to Pluvicto treatment site to get incidence rate information

In [34]:
# Create state-county name for lookup prostate cancer profile
state_cancer_profile_filtered["Lookup"] = (
    state_cancer_profile_filtered["state"] + 
    state_cancer_profile_filtered["locale"]
)

/var/folders/st/218s55451fg4sfvmvvt7rz780000gn/T/ipykernel_20416/2936143665.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  state_cancer_profile_filtered["Lookup"] = (


In [35]:
# Create Lookup column only where County exists
pluvicto_hos["Lookup"] = np.where(
    pluvicto_hos["County"].notna() & (pluvicto_hos["County"] != ""),
    pluvicto_hos["State_Full"] + pluvicto_hos["County"] + " County",
    None
)

# Split pluvicto_hos into two groups
pluvicto_hos_with_lookup = pluvicto_hos[pluvicto_hos['Lookup'].notna()].copy()
pluvicto_hos_without_lookup = pluvicto_hos[pluvicto_hos['Lookup'].isna()].copy()

# For rows WITH Lookup - merge on Lookup
pluvicto_hos_with_lookup = pluvicto_hos_with_lookup.merge(
    state_cancer_profile_filtered[['Lookup', 'age_adjusted_incidence_raterate_note___cases_per_100_000']],
    on='Lookup',
    how='left'
)

# For rows WITHOUT Lookup - calculate state averages and merge on State_Full
# Calculate state-level averages
state_avg_incidence = state_cancer_profile_filtered.groupby('state')['age_adjusted_incidence_raterate_note___cases_per_100_000'].mean().reset_index()
state_avg_incidence = state_avg_incidence.rename(columns={
    'state': 'State_Full',
    'age_adjusted_incidence_raterate_note___cases_per_100_000': 'state_avg_incidence'
})

# Merge state averages to rows without Lookup
pluvicto_hos_without_lookup = pluvicto_hos_without_lookup.merge(
    state_avg_incidence,
    on='State_Full',
    how='left'
)

# Rename the incidence column in both DataFrames
pluvicto_hos_with_lookup = pluvicto_hos_with_lookup.rename(columns={
    'age_adjusted_incidence_raterate_note___cases_per_100_000': 'prostate_cancer_incidence'
})

pluvicto_hos_without_lookup = pluvicto_hos_without_lookup.rename(columns={
    'state_avg_incidence': 'prostate_cancer_incidence'
})

# Combine back together
pluvicto_hos = pd.concat([pluvicto_hos_with_lookup, pluvicto_hos_without_lookup], ignore_index=True)

In [39]:
# Extract the national incidence as a single value
national_incidence = state_cancer_profile_filtered[
    state_cancer_profile_filtered["full_locale"] == "US (SEER+NPCR)(1)"
]["age_adjusted_incidence_raterate_note___cases_per_100_000"].iloc[0]

# fill any missing value with this national value
pluvicto_hos['prostate_cancer_incidence'] = pluvicto_hos['prostate_cancer_incidence'].fillna(national_incidence)

In [41]:
pluvicto_hos['prostate_cancer_incidence'] = round(pluvicto_hos['prostate_cancer_incidence'],2)

Assign total weights to each treatment site and estimate Pluvicto demand per site per year

In [46]:
# Calculate total weight for each hospital
pluvicto_hos["Demand Weight"] = (
    pluvicto_hos["Designation"] * 
    pluvicto_hos["Number of Beds"] * 
    pluvicto_hos["prostate_cancer_incidence"]
)

# Calculate proportional demand (250,000 total vials distributed by weight)
total_weight = pluvicto_hos["Demand Weight"].sum()
pluvicto_hos["Demand per Year"] = ((pluvicto_hos["Demand Weight"] / total_weight) * 250000).round(0)

Overview of the cleaned Pluvicto treament site with estimated demand

In [88]:
pluvicto_hos

,Hospital (Novartis),Address,City,State,Zip,rpt_rec_num,Designation,LATITUDE,LONGITUDE,State_Full,Number of Beds,County,Lookup,prostate_cancer_incidence,Demand Weight,Demand per Year
0,MONUMENT HEALTH RAPID CITY HOSPITAL - FAIRMONT...,353 FAIRMONT BLVD MEDICAL IMAGING/ NUCLEAR MED...,RAPID CITY,SD,57701,793743.0,1,44.079142,-103.215297,South Dakota,385.0,Pennington,South DakotaPennington County,19.60,7546.00,171.0
1,CHEYENNE REGIONAL MEDICAL CENTER,BLDG WEST CAMPUS FL 1 214 E 23RD ST,CHEYENNE,WY,82001-3748,791460.0,1,41.143175,-104.785212,Wyoming,159.0,Laramie,WyomingLaramie County,25.80,4102.20,93.0
2,BILLINGS CLINIC,2800 10TH AVE N,BILLINGS,MT,59101,777548.0,1,45.772835,-108.500117,Montana,334.0,Yellowstone,MontanaYellowstone County,29.20,9752.80,221.0
3,KAISER PERMANENTE ROCK CREEK,280 EXEMPLA CIR,LAFAYETTE,CO,80026,795452.0,1,39.996426,-105.101503,Colorado,183.0,Boulder,ColoradoBoulder County,26.90,4922.70,111.0
4,UCHEALTH- UNIVERSITY OF COLORADO HOSPITAL,1635 AURORA CT,AURORA,CO,80045,791643.0,5,39.746989,-104.837035,Colorado,754.0,Adams,ColoradoAdams County,21.70,81809.00,1850.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
621,IMPRESSION IMAGING LLC - BOCA RATON,6853 SW 18TH ST SUITE M-101,BOCA RATON,FL,33433-7056,NaN,1,26.345366,-80.155903,Florida,360.0,NaN,None,19.81,7131.60,161.0
622,UNIVERSITY OF MIAMI SYLVESTER COMPREHENSIVE CA...,1192 EAST NEWPORT CENTER DRIVE,DEERFIELD BEACH,FL,33442,NaN,2,26.311818,-80.142902,Florida,360.0,NaN,None,19.81,14263.20,323.0
623,BAPTIST HEALTH CANCER CARE,1228 S PINE ISLAND ROAD SUITE NO. SUITE 130,PLANTATION,FL,33324,NaN,1,26.114942,-80.272862,Florida,360.0,NaN,None,19.81,7131.60,161.0
624,UNIVERSITY OF MIAMI HOSPITAL AND CLINICS- SOLE...,BLDG UHEALTH - SOLE MIA STE 1060M FL 1 2111 SO...,NORTH MIAMI,FL,33181-2492,NaN,2,25.897716,-80.158404,Florida,360.0,NaN,None,19.81,14263.20,323.0


# Potential Production Location Cleaning and Expenditure Estimation

## Production Location Selection

The next step in preparing for the optimization problem is to select potential production locations. Due to the short half-life of radiopharmaceuticals (Pluvicto's half-life is approximately 6.65 days), the production facility must be strategically positioned to minimize shipping times and ensure timely delivery to end users. The production site should also be situated near major transportation infrastructure to facilitate timely distribution of the final product. Given these operational constraints, potential production locations are likely to be in close proximity to major U.S. cities, where there is a higher concentration of academic hospitals and NCI-designated cancer centers.

In this project, **100 potential production locations** were selected from the top 100 metropolitan and micropolitan statistical areas, as defined by the Core Based Statistical Areas ([CBSA](https://catalog.data.gov/dataset/core-based-statistical-areas1#:~:text=The%20Core%20Based%20Statistical%20Areas%20dataset%20was%20updated,Transportation%20Statistics%20%28BTS%29%20National%20Transportation%20Atlas%20Database%20%28NTAD%29.)), ranked by total population in descending order. Two are subsequently removed from the dataset due to lack of geographical centroid data. After the exclusion, in total, **98 potential production** locations are selected for this project. This approach ensures that production sites are located near large patient populations and can serve a substantial number of end users. The specific geographic coordinates of each potential production site are assigned as the **centroid of its respective CBSA** using the shape file published by the U.S. Census Bureau ([Tiger/Line Shapefile](https://catalog.data.gov/dataset/tiger-line-shapefile-2020-nation-u-s-core-based-statistical-areas-cbsa?from_hint=eyJxIjoiY2JzYSJ9)). It is important to note that this project is not intended to identify exact production site locations, as such decisions are highly dependent on other factors including talent acquisition, corporate strategy, local policy incentives, tax incentives, and other site-specific considerations that are beyond the scope of this model. Rather, this analysis aims to identify **regional clusters or metropolitan areas** where a production facility could feasibly be located, with final site selection to be determined by these additional criteria.

## Capital Expenditure Estimation

Land acquisition costs and construction expenditures are highly location-dependent. To estimate these costs, this project utilized **national average construction costs for advanced manufacturing facilities**, which range from **450 to 800 USD** per square foot for sectors such as precision machining, electronics, medical devices, batteries, and energy components ([Terrapin Construction Group, 2026](https://terrapincg.com/news/manufacturing-facility-construction-cost-2026)). The **midpoint of this range, $650 per square foot**, was adopted as the national baseline construction cost.

To account for geographic variation in construction costs, this baseline was scaled using **median CBSA-level ratios of local construction worker wages to national average construction worker wages**, as reported by the **Bureau of Labor Statistics (BLS) Occupational Employment and Wage Statistics (OEWS)** ([BLS OEWS Data](https://lmi.sc.gov/Data-Hub/BLS-Data/OEWS)). Since land acquisition costs typically represent a relatively small portion of total upfront investment, this approach effectively captures the primary drivers of geographic cost variation—namely, labor and material costs—while providing a reasonable approximation of initial capital expenditures across different regions.

In [70]:
potential_loc = pd.read_excel("Potential Location.xlsx")
salary = pd.read_excel("MSA_M2025_dl.xlsx")
centroid = gpd.read_file("tl_2020_us_cbsa.shp")

Overview of each dataset

In [71]:
potential_loc.head(5)

,CBSA,NAME,TOT_POP
0,35620,"New York-Newark-Jersey City, NY-NJ",20112448
1,31080,"Los Angeles-Long Beach-Anaheim, CA",12844441
2,16980,"Chicago-Naperville-Elgin, IL-IN",9434123
3,19100,"Dallas-Fort Worth-Arlington, TX",8477157
4,26420,"Houston-Pasadena-The Woodlands, TX",7904627


In [72]:
salary.head(5)

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.32,29.92,45.19,24900,30980,44340,62230,93990,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,45.9,64.34,92.98,47790,64990,95480,133840,193390,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.7,113.51,187.68,88400,121980,165770,236100,390370,NaN,NaN
3,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,41.44,67.13,101.14,43770,57740,86200,139630,210360,NaN,NaN
4,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,...,52.45,74.4,95.07,71010,85770,109100,154750,197740,NaN,NaN


In [73]:
centroid.head(5)

,CSAFP,CBSAFP,GEOID,NAME,NAMELSAD,LSAD,MEMI,MTFCC,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,122,12020,12020,"Athens-Clarke County, GA","Athens-Clarke County, GA Metro Area",M1,1,G3110,2654607902,26109459,+33.9439840,-083.2138965,"POLYGON ((-83.53738 33.96591, -83.53184 33.968..."
1,122,12060,12060,"Atlanta-Sandy Springs-Alpharetta, GA","Atlanta-Sandy Springs-Alpharetta, GA Metro Area",M1,1,G3110,22495780629,386874693,+33.6937280,-084.3999113,"POLYGON ((-85.33823 33.65312, -85.33842 33.654..."
2,428,12100,12100,"Atlantic City-Hammonton, NJ","Atlantic City-Hammonton, NJ Metro Area",M1,1,G3110,1438774368,301270979,+39.4693555,-074.6337591,"POLYGON ((-74.85675 39.42076, -74.8567 39.4208..."
3,426,12120,12120,"Atmore, AL","Atmore, AL Micro Area",M2,2,G3110,2448595161,20024887,+31.1222867,-087.1684097,"POLYGON ((-87.61542 31.041, -87.61542 31.04116..."
4,258,12140,12140,"Auburn, IN","Auburn, IN Micro Area",M2,2,G3110,939731962,2657419,+41.3967596,-085.0026969,"POLYGON ((-85.19295 41.38001, -85.19296 41.381..."


Merge CBSA and TIGER shapefile to get geographical centroid for each potential production site

In [74]:
# Convert both columns to string
potential_loc['CBSA'] = potential_loc['CBSA'].astype(str)
centroid['CBSAFP'] = centroid['CBSAFP'].astype(str)

# Merge CBSA and TIGER Shapefile on CBSA and CBSAFP
potential_loc = potential_loc.merge(
    centroid[["CBSAFP", "INTPTLAT", "INTPTLON"]],
    left_on="CBSA",
    right_on="CBSAFP",
    how="left"
)

Prepare salary data to include only construction worker wages and merge with potential locations to get construction worker salary data

In [77]:
salary = salary[salary["OCC_TITLE"] == "Construction and Extraction Occupations"]

In [78]:
salary["AREA"] = salary["AREA"].astype(str)

# Merge CBSA and salary on CBSA code
potential_loc = potential_loc.merge(
    salary[["AREA", "H_MEDIAN"]],
    left_on="CBSA",
    right_on="AREA",
    how="left"
)

In [81]:
# Drop reference columns CBSAFP and AREA, then drop 2 rows without geographical centroid values
potential_loc = potential_loc.drop(["CBSAFP", "AREA"], axis=1)
potential_loc = potential_loc.dropna()

Calculate factors to scale national construction cost and per-square-foot construction cost for each potential production site

In [85]:
# Calculate the national median salary and calculate ratio of local median salary to national median salary as the adjustment factor and 
# scale up the national mean construction cost per square foot
median_national_salary = salary["H_MEDIAN"].median()
potential_loc["Cost Level"] = round((potential_loc["H_MEDIAN"] / median_national_salary) * 650 , 2)

In [86]:
potential_loc

,CBSA,NAME,TOT_POP,INTPTLAT,INTPTLON,H_MEDIAN,Cost Level
0,35620,"New York-Newark-Jersey City, NY-NJ",20112448,+40.7749300,-073.8737580,36.56,856.051873
1,31080,"Los Angeles-Long Beach-Anaheim, CA",12844441,+34.1087033,-118.1827533,33.47,783.699568
2,16980,"Chicago-Naperville-Elgin, IL-IN",9434123,+41.8235214,-087.8282967,40.49,948.072767
3,19100,"Dallas-Fort Worth-Arlington, TX",8477157,+32.8491708,-096.9704894,24.06,563.364553
4,26420,"Houston-Pasadena-The Woodlands, TX",7904627,+29.7495926,-095.3536422,24,561.959654
...,...,...,...,...,...,...,...
95,27140,"Jackson, MS",609847,+32.4282801,-090.2023875,22.74,532.456772
96,44060,"Spokane-Spokane Valley, WA",608012,+48.0916180,-117.6634720,30.55,715.32781
97,45780,"Toledo, OH",599376,+41.5553128,-083.5170883,31.23,731.25
98,16860,"Chattanooga, TN-GA",594530,+35.0493612,-085.3611582,24.38,570.857349


## Save cleaned dataset for optimization model

In [89]:
pluvicto_hos.to_csv("pluvicto treament sites.csv", index = False)
potential_loc.to_csv("potential locations.csv", index=False)

# Distance Matrix Calculation

The next step in the analysis is to compute the distance/time matrix between production sites and treatment sites. To reduce computational burden, both in the API calls for the matrix and in the subsequent optimization solver, the number of potential production sites is limited to **50**.

In [4]:
treatment_sites  = pd.read_csv("pluvicto treament sites.csv")
production_sites = pd.read_csv("potential locations.csv")

In [5]:
# limit production sites to the first 50 locations
production_sites = production_sites.iloc[:49, :]

In [6]:
treatment_sites.head(5)

,Hospital (Novartis),Address,City,State,Zip,rpt_rec_num,Designation,LATITUDE,LONGITUDE,State_Full,Number of Beds,County,Lookup,prostate_cancer_incidence,Demand Weight,Demand per Year
0,MONUMENT HEALTH RAPID CITY HOSPITAL - FAIRMONT...,353 FAIRMONT BLVD MEDICAL IMAGING/ NUCLEAR MED...,RAPID CITY,SD,57701,793743.0,1,44.079142,-103.215297,South Dakota,385.0,Pennington,South DakotaPennington County,19.6,7546.0,171.0
1,CHEYENNE REGIONAL MEDICAL CENTER,BLDG WEST CAMPUS FL 1 214 E 23RD ST,CHEYENNE,WY,82001-3748,791460.0,1,41.143175,-104.785212,Wyoming,159.0,Laramie,WyomingLaramie County,25.8,4102.2,93.0
2,BILLINGS CLINIC,2800 10TH AVE N,BILLINGS,MT,59101,777548.0,1,45.772835,-108.500117,Montana,334.0,Yellowstone,MontanaYellowstone County,29.2,9752.8,221.0
3,KAISER PERMANENTE ROCK CREEK,280 EXEMPLA CIR,LAFAYETTE,CO,80026,795452.0,1,39.996426,-105.101503,Colorado,183.0,Boulder,ColoradoBoulder County,26.9,4922.7,111.0
4,UCHEALTH- UNIVERSITY OF COLORADO HOSPITAL,1635 AURORA CT,AURORA,CO,80045,791643.0,5,39.746989,-104.837035,Colorado,754.0,Adams,ColoradoAdams County,21.7,81809.0,1850.0


In [8]:
production_sites.head(5)

,CBSA,NAME,TOT_POP,INTPTLAT,INTPTLON,H_MEDIAN,Cost Level
0,35620,"New York-Newark-Jersey City, NY-NJ",20112448,40.774930,-73.873758,36.56,856.051873
1,31080,"Los Angeles-Long Beach-Anaheim, CA",12844441,34.108703,-118.182753,33.47,783.699568
2,16980,"Chicago-Naperville-Elgin, IL-IN",9434123,41.823521,-87.828297,40.49,948.072767
3,19100,"Dallas-Fort Worth-Arlington, TX",8477157,32.849171,-96.970489,24.06,563.364553
4,26420,"Houston-Pasadena-The Woodlands, TX",7904627,29.749593,-95.353642,24.00,561.959654


In [9]:
# Create coordinate lists
# For treatment sites
hospital_coords = treatment_sites[['LONGITUDE', 'LATITUDE']].values.tolist()

# For production sites
production_coords = production_sites[['INTPTLON', 'INTPTLAT']].values.tolist()

Request distance calculation via OSRM API call

In [10]:
# Initialize client
client = OSRM(base_url="http://router.project-osrm.org")

def get_chunked_matrix(origins, destinations, origin_chunk=3, dest_chunk=20):
    n_origins = len(origins)
    n_dests = len(destinations)
    
    duration_mat = np.full((n_origins, n_dests), np.nan)
    distance_mat = np.full((n_origins, n_dests), np.nan)
    
    for o_start in range(0, n_origins, origin_chunk):
        o_end = min(o_start + origin_chunk, n_origins)
        origin_slice = origins[o_start:o_end]
        
        for d_start in range(0, n_dests, dest_chunk):
            d_end = min(d_start + dest_chunk, n_dests)
            dest_slice = destinations[d_start:d_end]
            
            locations = origin_slice + dest_slice
            src_indices = list(range(len(origin_slice)))
            dst_indices = list(range(len(origin_slice), len(origin_slice) + len(dest_slice)))
            
            matrix = client.matrix(
                locations=locations,
                profile='driving',
                sources=src_indices,
                destinations=dst_indices
            )
            
            for oi, o_idx in enumerate(range(o_start, o_end)):
                for di, d_idx in enumerate(range(d_start, d_end)):
                    duration_mat[o_idx, d_idx] = matrix.durations[oi][di]
                    distance_mat[o_idx, d_idx] = matrix.distances[oi][di]
            
            time.sleep(0.5)
        
        print(f"Completed {o_end}/{n_origins} origins")
    
    return duration_mat, distance_mat

# Calculate matrices
duration_matrix, distance_matrix = get_chunked_matrix(
    origins=production_coords,
    destinations=hospital_coords,
    origin_chunk=3,
    dest_chunk=20
)

# Convert to DataFrames with proper labels
duration_df = pd.DataFrame(
    duration_matrix,
    index=production_sites['NAME'].tolist(),
    columns=treatment_sites['Hospital (Novartis)'].tolist()
)

distance_df = pd.DataFrame(
    distance_matrix,
    index=production_sites['NAME'].tolist(),
    columns=treatment_sites['Hospital (Novartis)'].tolist()
)

Completed 3/49 origins
Completed 6/49 origins
Completed 9/49 origins
Completed 12/49 origins
Completed 15/49 origins
Completed 18/49 origins
Completed 21/49 origins
Completed 24/49 origins
Completed 27/49 origins
Completed 30/49 origins
Completed 33/49 origins
Completed 36/49 origins
Completed 39/49 origins
Completed 42/49 origins
Completed 45/49 origins
Completed 48/49 origins
Completed 49/49 origins


In [11]:
# Convert duration from seconds to hours
duration_df = duration_df / 3600

In [12]:
duration_df

,MONUMENT HEALTH RAPID CITY HOSPITAL - FAIRMONT BLVD,CHEYENNE REGIONAL MEDICAL CENTER,BILLINGS CLINIC,KAISER PERMANENTE ROCK CREEK,UCHEALTH- UNIVERSITY OF COLORADO HOSPITAL,NATIONAL JEWISH HEALTH,ST ANTHONY HOSPITAL,ADVENT HEALTH PORTER,BOZEMAN HEALTH DEACONESS REGIONAL MEDICAL CENTER,AVERA MCKENNAN HOSPITAL,...,FLORIDA CANCER SPECIALISTS & RESEARCH INSTITUTE - GLADIOLUS,FLORIDA ONCOLOGY & HEMATOLOGY,FLORIDA THERANOSTICS - INDEPENDENT PRACTICE,MOBILE THERAPY UNIT- FLORIDA THERANOSTICS,IMPRESSION IMAGING LLC,IMPRESSION IMAGING LLC - BOCA RATON,UNIVERSITY OF MIAMI SYLVESTER COMPREHENSIVE CANCER CENTER,BAPTIST HEALTH CANCER CARE,UNIVERSITY OF MIAMI HOSPITAL AND CLINICS- SOLE MIA,BRAMAN COMPREHENSIVE CANCER CENTER- MOUNT SINAI MEDICAL CENTER
"New York-Newark-Jersey City, NY-NJ",30.907694,31.572639,36.758417,32.462667,32.379972,32.397250,32.581472,32.519917,39.100000,25.593917,...,24.401750,24.524278,22.896361,22.896361,23.777500,23.614917,23.729944,23.988944,24.384556,24.459028
"Los Angeles-Long Beach-Anaheim, CA",23.460389,19.010167,21.770194,18.094139,18.008889,17.846250,17.602139,17.820806,19.380417,28.982333,...,46.770500,46.893028,46.526167,46.526167,47.407306,47.244722,47.359750,47.618750,48.014361,48.088833
"Chicago-Naperville-Elgin, IL-IN",15.914944,16.815611,21.765667,17.705639,17.622944,17.640222,17.824444,17.762889,24.107250,10.601167,...,24.234056,24.356583,23.989722,23.989722,24.870861,24.708278,24.823306,25.082306,25.477917,25.552389
"Dallas-Fort Worth-Arlington, TX",18.859778,15.863694,23.166472,14.610833,14.114056,14.139694,14.299500,14.012500,25.508056,14.783000,...,23.079806,23.202333,22.835472,22.835472,23.716611,23.554028,23.669056,23.928056,24.323667,24.398139
"Houston-Pasadena-The Woodlands, TX",23.217333,20.312472,27.615250,19.059611,18.562833,18.588472,18.748278,18.461278,29.956833,19.140556,...,20.428139,20.550667,20.183806,20.183806,21.064944,20.902361,21.017389,21.276389,21.672000,21.746472
"Atlanta-Sandy Springs-Roswell, GA",26.960889,26.178389,32.829750,25.576361,25.100028,25.316917,25.543667,25.392417,35.171333,21.552972,...,10.741278,10.863806,10.496944,10.496944,11.378083,11.215500,11.330528,11.589528,11.985139,12.059611
"Washington-Arlington-Alexandria, DC-VA-MD-WV",29.123472,29.788417,34.974194,30.498139,30.021806,30.238694,30.465444,30.314194,37.315778,23.809694,...,19.582528,19.705056,18.077139,18.077139,18.958278,18.795694,18.910722,19.169722,19.565333,19.639806
"Miami-Fort Lauderdale-West Palm Beach, FL",39.331639,38.549139,45.200500,37.947111,37.470778,37.687667,37.914417,37.763167,47.542083,33.923722,...,2.991194,2.428417,2.307806,2.307806,1.193056,1.416361,1.379222,1.026750,1.413056,1.427861
"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",29.967944,30.632889,35.818667,31.522917,31.389472,31.457500,31.641722,31.580167,38.160250,24.654167,...,22.129194,22.251722,20.623806,20.623806,21.504944,21.342361,21.457389,21.716389,22.112000,22.186472
"Phoenix-Mesa-Chandler, AZ",23.252806,17.910806,23.003083,16.599778,16.089194,16.114833,16.186917,15.987639,20.613306,27.106389,...,40.122611,40.245139,39.878278,39.878278,40.759417,40.596833,40.711861,40.970861,41.366472,41.440944


In [13]:
# Convert distance from meters to miles
distance_df = distance_df / 1609.344

In [14]:
distance_df

,MONUMENT HEALTH RAPID CITY HOSPITAL - FAIRMONT BLVD,CHEYENNE REGIONAL MEDICAL CENTER,BILLINGS CLINIC,KAISER PERMANENTE ROCK CREEK,UCHEALTH- UNIVERSITY OF COLORADO HOSPITAL,NATIONAL JEWISH HEALTH,ST ANTHONY HOSPITAL,ADVENT HEALTH PORTER,BOZEMAN HEALTH DEACONESS REGIONAL MEDICAL CENTER,AVERA MCKENNAN HOSPITAL,...,FLORIDA CANCER SPECIALISTS & RESEARCH INSTITUTE - GLADIOLUS,FLORIDA ONCOLOGY & HEMATOLOGY,FLORIDA THERANOSTICS - INDEPENDENT PRACTICE,MOBILE THERAPY UNIT- FLORIDA THERANOSTICS,IMPRESSION IMAGING LLC,IMPRESSION IMAGING LLC - BOCA RATON,UNIVERSITY OF MIAMI SYLVESTER COMPREHENSIVE CANCER CENTER,BAPTIST HEALTH CANCER CARE,UNIVERSITY OF MIAMI HOSPITAL AND CLINICS- SOLE MIA,BRAMAN COMPREHENSIVE CANCER CENTER- MOUNT SINAI MEDICAL CENTER
"New York-Newark-Jersey City, NY-NJ",1714.513491,1746.359883,2051.200986,1787.162036,1789.935837,1788.495499,1798.552392,1795.850980,2191.912357,1378.317501,...,1277.097563,1289.589485,1208.533601,1208.533601,1256.857639,1248.233628,1253.704304,1266.924536,1283.584057,1290.580696
"Los Angeles-Long Beach-Anaheim, CA",1320.282861,1098.984182,1233.919286,1026.602703,1024.230308,1014.152599,1004.807797,1015.189481,1092.037439,1667.647998,...,2661.961644,2674.453690,2652.183436,2652.183436,2700.507474,2691.883463,2697.354015,2710.574309,2727.233892,2734.230221
"Chicago-Naperville-Elgin, IL-IN",912.870896,951.155813,1249.558391,991.958090,994.731891,993.291428,1003.348321,1000.647034,1390.269762,576.674782,...,1283.623576,1296.115622,1273.845368,1273.845368,1322.169344,1313.545333,1319.016009,1332.236178,1348.895761,1355.892401
"Dallas-Fort Worth-Arlington, TX",1051.985778,865.146296,1315.777609,790.308225,784.775909,781.968926,791.710784,779.062463,1456.489104,833.213347,...,1251.128410,1263.620332,1241.350078,1241.350078,1289.674240,1281.050105,1286.520781,1299.741075,1316.400533,1323.397173
"Houston-Pasadena-The Woodlands, TX",1297.727459,1116.488830,1567.120205,1041.650697,1036.118443,1033.311399,1043.053256,1030.404935,1707.831576,1078.955028,...,1116.376983,1128.868968,1106.598714,1106.598714,1154.922813,1146.298740,1151.769479,1164.989648,1181.649169,1188.645746
"Atlanta-Sandy Springs-Roswell, GA",1512.117049,1443.003485,1827.961890,1425.380466,1397.029162,1406.227817,1417.925130,1410.818010,1968.673447,1169.966210,...,588.136284,600.628206,578.358014,578.358014,626.682114,618.058041,623.528779,636.748949,653.408407,660.405047
"Washington-Arlington-Alexandria, DC-VA-MD-WV",1603.031421,1634.877814,1939.718917,1685.354716,1657.003599,1666.202192,1677.899504,1670.792261,2080.430287,1266.835431,...,1038.183881,1050.675865,969.619920,969.619920,1017.943957,1009.319947,1014.790685,1028.010854,1044.670375,1051.667015
"Miami-Fort Lauderdale-West Palm Beach, FL",2165.349049,2096.235485,2481.193890,2078.612466,2050.261162,2059.459817,2071.157130,2064.050010,2621.905571,1823.198148,...,126.042785,100.412839,84.958530,84.958530,28.137614,39.524862,37.562883,21.645714,32.795102,35.244733
"Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",1668.233330,1700.079660,2004.920825,1740.881875,1712.864372,1742.215151,1752.272044,1749.570819,2145.632009,1332.037153,...,1168.696376,1181.188360,1100.132414,1100.132414,1148.456452,1139.832441,1145.303179,1158.523349,1175.182870,1182.179509
"Phoenix-Mesa-Chandler, AZ",1279.190030,976.116107,1238.967120,896.621543,874.680305,871.873322,840.378688,868.966859,1097.085272,1427.932934,...,2277.054999,2289.546859,2267.276605,2267.276605,2315.600829,2306.976818,2312.447370,2325.667663,2342.327246,2349.323700


In [15]:
duration_df.to_csv("duration.csv",index = True)
distance_df.to_csv("distance.csv",index = True)